# Example — Account transformation

Reads from the PostgreSQL staging tables populated by `extract/extract.py`.
SQL lives in `../../sql/example_accounts.sql` — edit the query there, not here.

Reusable pattern: **Setup -> Load -> Explore -> Build target -> Export**.
Copy this notebook per object and adapt the field mapping in section 4.

## 1 - Setup

In [ ]:
import os
import pathlib
import csv
from datetime import datetime

import pandas as pd
import psycopg2
from dotenv import load_dotenv

# Anchor to this notebook's location — works regardless of kernel cwd.
# VSCode injects __vsc_ipynb_file__ with the absolute path to the notebook file.
# example_transform.ipynb is at: transform/notebooks/example_object/
# parents[0] = notebooks/   parents[1] = transform/   parents[2] = repo root
NOTEBOOK_DIR = pathlib.Path(globals()['__vsc_ipynb_file__']).parent
REPO_ROOT    = NOTEBOOK_DIR.parents[2]

ENV_FILE = REPO_ROOT / "extract" / ".env"
load_dotenv(ENV_FILE)

DATABASE_URL = os.environ["DATABASE_URL"]
print("Connected to:", DATABASE_URL.split("@")[-1])  # print host only, not password

## 2 - Load source data

In [ ]:
SQL_FILE = REPO_ROOT / "transform" / "sql" / "example_accounts.sql"
query = SQL_FILE.read_text(encoding="utf-8")
print(f"Loaded query from: {SQL_FILE}")
print("-" * 60)
print(query)

In [ ]:
conn = psycopg2.connect(DATABASE_URL)
try:
    source_df = pd.read_sql_query(query, conn)
finally:
    conn.close()

print(f"Rows returned: {len(source_df)}")
source_df.head()

## 3 - Explore source data

In [ ]:
source_df.info()

In [ ]:
# Null counts per column — spot missing/required-field gaps before building the target frame
source_df.isnull().sum().sort_values(ascending=False)

In [ ]:
# Duplicate source_id check (excluding first occurrence) — should be 0
source_df['source_id'].duplicated().sum()

## 4 - Build target DataFrame

Direct field mappings from source columns to target Salesforce fields.
Replace this block with the real mapping for the object being migrated —
this is only a minimal illustrative example.

In [ ]:
source_df = source_df.reset_index(drop=True)

target_df = pd.DataFrame({
    'Legacy_Record_ID__c': source_df['source_id'].astype(str),
    'Name':                source_df['name'],
    'BillingCity':         source_df['billing_city'],
    'BillingCountry':      source_df['billing_country'],
})

In [ ]:
# Example: enforce a max length constraint on a target field
target_df.loc[target_df['Name'].str.len() > 80, 'Name'] = target_df['Name'].str.slice(0, 80)

In [ ]:
# Example: date reformatting to Salesforce's expected ISO-8601 shape
target_df['CreatedDate'] = pd.to_datetime(
    source_df['source_created_at'], errors='coerce'
).dt.strftime('%Y-%m-%dT%H:%M:%S.000Z')

In [ ]:
pd.set_option('display.max_columns', None)
target_df.head()

## 5 - Export

In [ ]:
output_path = REPO_ROOT / "transform" / "output" / "account" / "accounts.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

target_df.to_csv(output_path, index=False, quoting=csv.QUOTE_ALL)
print(f"Wrote {len(target_df)} rows -> {output_path}")